In [1]:
!apt-get install redis-server -y
!pip install redis dash pandas seaborn matplotlib plotly
!redis-server --daemonize yes

'apt-get' is not recognized as an internal or external command,
operable program or batch file.


Looking in indexes: https://pypi.org/simple, https://pypi.ngc.nvidia.com


'redis-server' is not recognized as an internal or external command,
operable program or batch file.


In [7]:
import redis
import pandas as pd
import pickle

#Connect to Redis
r = redis.Redis(host='localhost', port = 6379, db = 0)

#Load and Cache
def get_cached_dataset(key='restaurant_df', path='./data/Coffee Shop Sales.csv'):
  if r.exists(key):
    print("Loading from Redis cache...")
    return pickle.loads(r.get(key))
  else:
    print("Loading from csv and caching in Redis....")
    df = pd.read_csv(path)
    r.set(key, pickle.dumps(df))
    return df

#Load Data
df = get_cached_dataset()

#Preview
df.columns = df.columns.str.strip()
df['date'] = df['transaction_date'] + '-' +  df['transaction_time']
df.drop(['transaction_date', 'transaction_time'], axis=1, inplace=True)
df['date'] = pd.to_datetime(df['date'], format = "mixed")
df.set_index('date', inplace=True)
df.head()

Loading from Redis cache...


,transaction_id,transaction_qty,store_id,store_location,product_id,unit_price,product_category,product_type,product_detail
date,,,,,,,,,
2023-01-01 07:06:11,1,2,5,Lower Manhattan,32,3.0,Coffee,Gourmet brewed coffee,Ethiopia Rg
2023-01-01 07:08:56,2,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg
2023-01-01 07:14:04,3,2,5,Lower Manhattan,59,4.5,Drinking Chocolate,Hot chocolate,Dark chocolate Lg
2023-01-01 07:20:24,4,1,5,Lower Manhattan,,2.0,Coffee,Drip coffee,Our Old Time Diner Blend Sm
2023-01-01 07:22:41,5,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 149116 entries, 2023-01-01 07:06:11 to 2023-06-30 20:57:19
Data columns (total 9 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   transaction_id    149116 non-null  int64  
 1   transaction_qty   149116 non-null  int64  
 2   store_id          149116 non-null  int64  
 3   store_location    149116 non-null  object 
 4   product_id        149116 non-null  object 
 5   unit_price        149116 non-null  float64
 6   product_category  149116 non-null  object 
 7   product_type      149116 non-null  object 
 8   product_detail    149116 non-null  object 
dtypes: float64(1), int64(3), object(5)
memory usage: 11.4+ MB


In [9]:
# add columns like month, day number, day name, is weekend
df['Month'] = df.index.month
df['Day'] = df.index.day
df['Day_name'] = df.index.day_name()

df['is_weekend'] = df.index.weekday > 4

df['Hour'] = df.index.hour

df['revenue'] = df['transaction_qty'] * df['unit_price']


daily_sales = df['transaction_qty'].resample('D').sum()
missing_days = daily_sales.isna().sum()


df.head()

,transaction_id,transaction_qty,store_id,store_location,product_id,unit_price,product_category,product_type,product_detail,Month,Day,Day_name,is_weekend,Hour,revenue
date,,,,,,,,,,,,,,,
2023-01-01 07:06:11,1,2,5,Lower Manhattan,32,3.0,Coffee,Gourmet brewed coffee,Ethiopia Rg,1,1,Sunday,True,7,6.0
2023-01-01 07:08:56,2,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg,1,1,Sunday,True,7,6.2
2023-01-01 07:14:04,3,2,5,Lower Manhattan,59,4.5,Drinking Chocolate,Hot chocolate,Dark chocolate Lg,1,1,Sunday,True,7,9.0
2023-01-01 07:20:24,4,1,5,Lower Manhattan,,2.0,Coffee,Drip coffee,Our Old Time Diner Blend Sm,1,1,Sunday,True,7,2.0
2023-01-01 07:22:41,5,2,5,Lower Manhattan,57,3.1,Tea,Brewed Chai tea,Spicy Eye Opener Chai Lg,1,1,Sunday,True,7,6.2


In [11]:
daily_summary = df.resample('D').agg({
    'transaction_qty': 'sum',
    'revenue': 'sum'
})

daily_summary

,transaction_qty,revenue
date,,
2023-01-01,802,2508.20
2023-01-02,796,2466.30
2023-01-03,968,3040.25
2023-01-04,1198,3699.90
2023-01-05,1515,4731.45
...,...,...
2023-12-02,872,2894.00
2023-12-03,947,3088.33
2023-12-04,1239,4040.18


In [13]:
df.isnull().sum()

transaction_id      0
transaction_qty     0
store_id            0
store_location      0
product_id          0
unit_price          0
product_category    0
product_type        0
product_detail      0
Month               0
Day                 0
Day_name            0
is_weekend          0
Hour                0
revenue             0
dtype: int64